In [1]:
library(Seurat)
library(DoubletFinder)
library(qs)
library(ggplot2)
library(glue)
library(scico)
library(RColorBrewer)
library(patchwork)
library(ggsci)
library(dplyr)

Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.4.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘qs’ was built under R version 4.4.3”
qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘patchwork’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
all_doublet_detected_objs <- list.files("/data1/deyk/harry/RA_Xenium/data/Ian_old_scRNA_07062025/doublet_removed_seurat")
all_doublet_detected_objs <- lapply(all_doublet_detected_objs, function(file){
    qread(glue("/data1/deyk/harry/RA_Xenium/data/Ian_old_scRNA_07062025/doublet_removed_seurat/{file}"))
})

In [3]:
plot_doublet_and_remove <- lapply(all_doublet_detected_objs, function(obj){
    experiment <- obj$orig.ident[1]
    doublet_column_name <- colnames(obj@meta.data)[grepl("DF.class", colnames(obj@meta.data))]
    n_doublets <- sum(obj[[doublet_column_name]] == "Doublet")
    DimPlot(obj, group.by=doublet_column_name, pt.size=0.1, alpha=0.8) +
    theme_classic() +
    theme(aspect.ratio=1,
          axis.ticks=element_blank(),
          axis.text=element_blank(),
          legend.key.size=unit(5, "mm")) +
    xlab("UMAP1") +
    ylab("UMAP2") +
    ggtitle(glue("{experiment}\n{n_doublets} doublets")) +
    scale_color_uchicago() -> doublet_plot

    obj <- subset(obj, subset=!!sym(doublet_column_name)!="Doublet")
    return(list(obj=obj, figure=doublet_plot))
})

In [4]:
doublet_figures <- wrap_plots(lapply(plot_doublet_and_remove, function(res){
    res[["figure"]]
}), ncol=3)

In [156]:
ggsave(plot=doublet_figures,
       filename="/data1/deyk/harry/RA_Xenium/results/figures/Ian_old_scrna_experiment/DoubletFinder/Doublets.png",
       height=20,
       width=15,
       dpi=500, 
       limitsize=FALSE)

In [5]:
doublet_removed_objs <- lapply(plot_doublet_and_remove, function(res){
    res[["obj"]]
})

In [6]:
plot_doublet_and_fb_signatures <- lapply(doublet_removed_objs, function(obj){
    experiment <- obj$orig.ident[1]

    DimPlot(obj, group.by="seurat_clusters", pt.size=0.1, alpha=0.8) +
    theme_classic() +
    theme(aspect.ratio=1,
          axis.ticks=element_blank(),
          axis.text=element_blank(),
          legend.key.size=unit(5, "mm")) +
    xlab("UMAP1") +
    ylab("UMAP2") +
    ggtitle(glue("{experiment}\nclusters")) +
    scale_color_manual(values=brewer.pal(name="Paired", n=12)) -> cluster_plot

    fibroblast_markers <- c("COL1A1", "COL1A2", "COL5A1", "LOXL1", "LUM", "FBLN1", "FBLN2", "SERPINH1", "FAP", "THY1")
    obj <- AddModuleScore(obj,
                          list("FB"=fibroblast_markers),
                          assay="SCT",
                          name="FB markers",
                          nbin=19)
    FeaturePlot(obj, features="FB markers1") +
    scale_color_scico(palette="vik", midpoint=0) +
    theme_classic() +
    theme(aspect.ratio=1,
          axis.ticks=element_blank(),
          axis.text=element_blank(),
          legend.key.size=unit(3, "mm")) +
    xlab("UMAP1") +
    ylab("UMAP2") +
    ggtitle("Fibroblast signature") -> fb_sig_plot
    return(list(obj=obj, plots=list(cluster_plot, fb_sig_plot)))
})

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Warning message:
“The `slot` argument of `FetchData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
Scale

In [216]:
options(repr.plot.width = 10, repr.plot.height = 5, repr.plot.res = 500)
fb_sig_plot_all <- wrap_plots(unlist(lapply(plot_doublet_and_fb_signatures, function(res){
    res[["plots"]]
}), recursive=FALSE), ncol=2)

In [217]:
ggsave(plot=fb_sig_plot_all,
       filename="/data1/deyk/harry/RA_Xenium/results/figures/Ian_old_scrna_experiment/FB_signature/FB_signature.png",
       height=60,
       width=15,
       dpi=500, 
       limitsize=FALSE)

# Get mean fibroblast marker expression and subset to macrophage clusters

In [7]:
doublet_removed_objs_seurat <- lapply(plot_doublet_and_fb_signatures, function(res){
    res[["obj"]]
})

In [15]:
lapply(doublet_removed_objs_seurat, function(obj){
    obj@meta.data %>%
        select(seurat_clusters, `FB markers1`) %>%
        group_by(seurat_clusters) %>%
        summarize(median_score=median(`FB markers1`)) -> median_fb_marker_score
    as.character(median_fb_marker_score$seurat_clusters[which(median_fb_marker_score$median_score > 0.2)])
}) -> fibroblast_clusters

In [10]:
lapply(doublet_removed_objs_seurat, function(obj){
    obj@meta.data %>%
        select(seurat_clusters, `FB markers1`) %>%
        group_by(seurat_clusters) %>%
        summarize(median_score=median(`FB markers1`)) -> median_fb_marker_score
    as.character(median_fb_marker_score$seurat_clusters[which(median_fb_marker_score$median_score < 0.1)])
}) -> macrophage_clusters

In [18]:
macrophage_cell_subset <- lapply(seq_along(doublet_removed_objs_seurat), function(idx){
    subset(doublet_removed_objs_seurat[[idx]], subset=seurat_clusters %in% macrophage_clusters[[idx]])
})

In [19]:
fibroblast_cell_subset <- lapply(seq_along(doublet_removed_objs_seurat), function(idx){
    subset(doublet_removed_objs_seurat[[idx]], subset=seurat_clusters %in% fibroblast_clusters[[idx]])
})

In [244]:
save_macrophage_cell_subset <- lapply(macrophage_cell_subset, function(obj){
    experiment <- obj$orig.ident[1]
    qsave(obj, glue("/data1/deyk/harry/RA_Xenium/data/Ian_old_scRNA_07062025/macrophage_seurat/{experiment}.qs"))
})

In [20]:
save_fibroblast_cell_subset <- lapply(fibroblast_cell_subset, function(obj){
    experiment <- obj$orig.ident[1]
    qsave(obj, glue("/data1/deyk/harry/RA_Xenium/data/Ian_old_scRNA_07062025/fibroblast_seurat/{experiment}.qs"))
})